# agentfix — tier 2: Kaggle fallback

**This notebook is UNTESTED on Kaggle.** It has never been run on Kaggle from the environment
that built this repo — there is no Kaggle access there. Every cell is built from commands
verified locally (`README.md`, `WORKSHOP.md`), but treat the Kaggle-specific parts as unverified
until someone runs it end to end. If Ollama does not come up here, fall back to tier 3: set
`MELLUM_MODEL=qwen2.5-coder:1.5b` and run everything locally with that ~1 GB model instead.

Use this notebook if your laptop cannot run the default 8 GB model (tier 1). Everything below
runs **inside** this Kaggle container — Ollama, the model, and the repo. There is no tunnel back
to your laptop, and no code changes: `MELLUM_BASE_URL` stays at its default
`http://localhost:11434/v1`, because inside this container that address *is* local.

**Before running this:**
1. Your Kaggle account must be **phone-verified** — this is required to enable internet access
   for a notebook. Do this well before the workshop, not on the day.
2. In Notebook Settings, turn **Internet: On**.
3. In Notebook Settings, select a **GPU accelerator** (e.g. GPU T4 x2). Free GPU quota is about
   30 hours/week per account.
4. **Edit `REPO_URL` in the next cell.** It is a placeholder, and the notebook stops with a clear
   error until you replace it with the URL you actually cloned this repo from.

Run the cells in order, top to bottom.

## 1. Set the repository URL and branch

Replace the value below with your own fork or clone URL — the one you or your instructor gave
you. There is no working default, because this repo has no canonical public home.

In [ ]:
REPO_URL = "https://github.com/REPLACE-ME/agentfix-workshop.git"  # <-- EDIT THIS LINE

# The branch matters as much as the URL. `main` is the STUDENT STARTING POINT: three pieces of
# the agent are deliberately stubbed, so `agentfix solve` on `main` prints an error and exits 1
# BY DESIGN. `solutions` is the complete implementation. Clone `solutions` so the demo cell at
# the bottom actually runs; `git checkout main` when you want to do the exercises yourself.
BRANCH = "solutions"

if "REPLACE-ME" in REPO_URL:
    raise SystemExit("Edit REPO_URL above before running the rest of this notebook.")
print(f"will clone branch {BRANCH!r} from {REPO_URL}")

## 2. Install and start Ollama

`OLLAMA_CONTEXT_LENGTH` is set in the **server's** environment here, which is the only place it
has any effect. Exporting it in your own shell does nothing, and Ollama's OpenAI-compatible
`/v1` endpoint silently drops a per-request `num_ctx`. See `Modelfile` for the measurements.

The readiness poll replaces a fixed `sleep`: on a cold Kaggle container the server can take
longer than any sleep you would be willing to hard-code, and a short sleep turns that into a
confusing connection error three cells later.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

import os
import subprocess
import time
import urllib.error
import urllib.request

server_env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="16384")
subprocess.Popen(["ollama", "serve"], env=server_env)


def wait_for_ollama(timeout_s: int = 180) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2):
                print("ollama is up")
                return
        except (urllib.error.URLError, OSError):
            time.sleep(1)
    raise RuntimeError(f"ollama did not become ready within {timeout_s}s")


wait_for_ollama()
!ollama --version

## 3. Pull the model

The GGUF is ~8 GB. Pulling it fresh every session wastes your GPU quota and the session's
network time. **The Kaggle Dataset trick:** after your first successful pull, save the model
as a private Kaggle Dataset (`ollama` stores models under `~/.ollama/models` — add that path
as a Dataset output, or `ollama pull` inside a session with the Dataset already attached as an
input and pointed at via `OLLAMA_MODELS`) and attach it as a Dataset input to future sessions
instead of re-pulling. This is worth doing once you've confirmed the pull below works, rather
than something to set up blind before you know the base path is right on this Kaggle image.

In [ ]:
!ollama pull hf.co/JetBrains/Mellum2-12B-A2.5B-Instruct-GGUF-Q4_K_M

## 4. Clone the repo and derive the model

`ollama create` bakes `num_ctx 16384` into a derived model named `agentfix-mellum2`, which is
what `LLMConfig` talks to. That is belt-and-braces with the server env in cell 2 — either alone
would do — but `agentfix doctor` checks for the derived model by name, so create it.

In [ ]:
!git clone --branch {BRANCH} {REPO_URL} agentfix-workshop
%cd agentfix-workshop
!git branch --show-current
!ollama create agentfix-mellum2 -f Modelfile
!curl -LsSf https://astral.sh/uv/install.sh | sh
!~/.local/bin/uv sync --extra dev

## 5. Preflight

Every check must read `[PASS]`. If `context window` reports anything under 16384 the
`ollama create` above did not take — re-run cell 4 rather than continuing, because a
4096-token window silently truncates the agent's history mid-run.

In [ ]:
!~/.local/bin/uv run agentfix doctor
!ollama ps

## 6. The workshop commands

Same commands as tiers 1 and 3 — no `MELLUM_BASE_URL` override needed, since Ollama is running
on `localhost` inside this same container.

`solve` works here because we cloned `solutions`. To do the exercises yourself run
`!git checkout main` first — then `exercises/` will be red (that is the point), and
`agentfix solve` will print an error and exit 1 until all three stages are finished.

In [ ]:
!~/.local/bin/uv run pytest exercises -q
!~/.local/bin/uv run agentfix solve tasks/workshop/01-shopcart --verbose